# AutoBot — Scorer Model Training (Qwen2.5-1.5B + LoRA SEQ_CLS)

**Model**: Qwen2.5-1.5B-Instruct with LoRA adapter for 3-class sequence classification

**Task**: Predict bottleneck risk band (0=low, 1=medium, 2=high) from GitHub issue snapshots

**Hardware**: A100 80GB GPU (Colab)

**Training specs** (from AutoBot_Ref_v5):
- LoRA r=16, alpha=32, targets: q_proj, v_proj, k_proj, o_proj
- Epochs: 4, Batch: 16, LR: 2e-4, Max seq: 1536 tokens
- 3x loss weight on high band (class 2) to handle imbalance
- days_open jitter ±3 for generalization

## 1. Install Dependencies

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 datasets accelerate bitsandbytes trl
!pip install -q scikit-learn matplotlib seaborn

## 2. Upload Data

Upload `train.jsonl`, `val.jsonl`, `test.jsonl` from `labelling/data/labeled/scorer/` to Colab.

Option A — Google Drive mount (recommended for large files):
```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/autobot/scorer'  # adjust path
```

Option B — Direct upload:

In [ ]:
import os

# --- Choose one option ---

# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/autobot/scorer'  # <-- adjust to your path

# Option B: Direct upload (uncomment below, comment out Option A)
# from google.colab import files
# uploaded = files.upload()  # upload train.jsonl, val.jsonl, test.jsonl
# DATA_DIR = '/content'

print(f"Data dir: {DATA_DIR}")
print(os.listdir(DATA_DIR))

## 3. Load & Prepare Dataset

In [ ]:
import json
import random
from collections import Counter

random.seed(42)

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val_raw = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))
test_raw = load_jsonl(os.path.join(DATA_DIR, 'test.jsonl'))

print(f"Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}")
print(f"Train band dist: {Counter(r['label']['band_name'] for r in train_raw)}")

In [ ]:
def apply_days_open_jitter(text: str, jitter_range: int = 3) -> str:
    """Apply ±3 day jitter to DAYS_OPEN in the prompt string.
    
    Prevents the model from learning a step function on exact T+7/T+15/etc values.
    Label stays unchanged — only the prompt number varies.
    """
    import re
    match = re.search(r'DAYS_OPEN: (\d+)', text)
    if match:
        original = int(match.group(1))
        jittered = max(1, original + random.randint(-jitter_range, jitter_range))
        text = text.replace(f'DAYS_OPEN: {original}', f'DAYS_OPEN: {jittered}')
    return text


def prepare_examples(records, apply_jitter=False):
    """Convert raw JSONL records to (text, label) pairs.
    
    Strips SNAPSHOT_TIER from input (not available at inference).
    """
    examples = []
    for r in records:
        text = r['input']
        label = r['label']['band']  # 0=low, 1=medium, 2=high
        
        # Remove SNAPSHOT_TIER from input (inference won't have it)
        text = re.sub(r'\| SNAPSHOT_TIER: T\+\d+\s*', '', text)
        
        if apply_jitter:
            text = apply_days_open_jitter(text)
        
        examples.append({'text': text, 'label': label})
    return examples

import re

train_data = prepare_examples(train_raw, apply_jitter=True)
val_data = prepare_examples(val_raw, apply_jitter=False)
test_data = prepare_examples(test_raw, apply_jitter=False)

print(f"Sample input (first 200 chars): {train_data[0]['text'][:200]}")
print(f"Sample label: {train_data[0]['label']}")

## 4. Load Tokenizer & Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
NUM_LABELS = 3  # low, medium, high
MAX_SEQ_LEN = 1536

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Qwen uses eos_token as pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")

## 5. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()  # save VRAM on A100
model.print_trainable_parameters()

## 6. Tokenize Dataset

In [ ]:
from datasets import Dataset

def tokenize_fn(examples):
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
    )
    tokens['labels'] = examples['label']
    return tokens

train_ds = Dataset.from_list(train_data).map(tokenize_fn, batched=True, remove_columns=['text'])
val_ds = Dataset.from_list(val_data).map(tokenize_fn, batched=True, remove_columns=['text'])
test_ds = Dataset.from_list(test_data).map(tokenize_fn, batched=True, remove_columns=['text'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

# Check token lengths
sample_lens = [len(tokenizer.encode(d['text'], truncation=False)) for d in train_data[:100]]
print(f"Token length stats (first 100): min={min(sample_lens)}, max={max(sample_lens)}, "
      f"mean={sum(sample_lens)/len(sample_lens):.0f}")
print(f"Exceeding {MAX_SEQ_LEN}: {sum(1 for l in sample_lens if l > MAX_SEQ_LEN)}/100")

## 7. Training Setup

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from transformers import Trainer, TrainingArguments

# --- Class weights: 3x on high band ---
# Band distribution: low ~39.5%, medium ~44.8%, high ~15.8%
# Apply 3x weight to high to get ~1,608 effective examples
class_weights = torch.tensor([1.0, 1.0, 3.0], dtype=torch.float32)


class WeightedTrainer(Trainer):
    """Trainer with per-class loss weighting."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc        = accuracy_score(labels, preds)
    f1_macro   = f1_score(labels, preds, average='macro')
    f1_per     = f1_score(labels, preds, average=None, labels=[0,1,2], zero_division=0)
    prec_per   = precision_score(labels, preds, average=None, labels=[0,1,2], zero_division=0)
    recall_per = recall_score(labels, preds, average=None, labels=[0,1,2], zero_division=0)

    return {
        # Overall
        'accuracy':       acc,
        'f1_macro':       f1_macro,
        # High band — primary go/no-go signals
        'recall_high':    recall_per[2],   # #1 priority — must be > 0.70
        'f1_high':        f1_per[2],       # #2 priority — must be > 0.65
        'precision_high': prec_per[2],
        # Medium band
        'f1_medium':      f1_per[1],
        'recall_medium':  recall_per[1],
        # Low band
        'precision_low':  prec_per[0],     # must stay > 0.70
        'f1_low':         f1_per[0],
    }


OUTPUT_DIR = '/content/scorer_checkpoints'

# Drive path for persistent storage
DRIVE_DIR = '/content/drive/MyDrive/autobot/scorer'
DRIVE_PLOTS = f'{DRIVE_DIR}/plots'
os.makedirs(DRIVE_PLOTS, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print(f"Training config: {training_args.num_train_epochs} epochs, "
      f"batch={training_args.per_device_train_batch_size}, "
      f"lr={training_args.learning_rate}, max_seq={MAX_SEQ_LEN}")
print(f"Class weights: {class_weights.tolist()} (3x on high band)")

## 8. Train

In [ ]:
trainer.train()

## 9. Evaluate on Test Set

In [ ]:
# Eval on test set
test_results = trainer.evaluate(test_ds)
print("\nTest Results:")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}")

# Detailed classification report
preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print("\nClassification Report:")
print(classification_report(
    labels, preds,
    target_names=['low', 'medium', 'high'],
    digits=3,
))

## 10. Visualizations (Presentation-Ready)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

BAND_LABELS = ['low', 'medium', 'high']
COLORS = ['#4CAF50', '#FFC107', '#F44336']  # green, amber, red

# Compute all per-band metrics for plots
per_prec = precision_score(labels, preds, average=None, labels=[0,1,2], zero_division=0)
per_rec = recall_score(labels, preds, average=None, labels=[0,1,2], zero_division=0)
per_f1 = f1_score(labels, preds, average=None, labels=[0,1,2], zero_division=0)

In [ ]:
# --- 10a. Confusion Matrix (normalized + raw counts) ---
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=BAND_LABELS, yticklabels=BAND_LABELS, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=BAND_LABELS, yticklabels=BAND_LABELS, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Normalized)')

fig.suptitle('Scorer Model — Test Set Confusion Matrix', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_PLOTS}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 10b. Per-Band Precision / Recall / F1 (grouped bar chart) ---
x = np.arange(len(BAND_LABELS))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width, per_prec, width, label='Precision', color='#2196F3')
bars2 = ax.bar(x, per_rec, width, label='Recall', color='#FF9800')
bars3 = ax.bar(x + width, per_f1, width, label='F1', color='#9C27B0')

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(BAND_LABELS)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Per-Band Precision / Recall / F1')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_PLOTS}/per_band_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 10c. High Band Focus — Recall is #1 priority ---
fig, ax = plt.subplots(figsize=(6, 4))
metrics = ['Recall', 'F1', 'Precision']
values = [per_rec[2], per_f1[2], per_prec[2]]
colors_h = ['#F44336', '#E91E63', '#FF5722']

bars = ax.barh(metrics, values, color=colors_h, height=0.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.15)
ax.set_title('High Band Metrics (Most Critical)', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_PLOTS}/high_band_focus.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 10d. Training Loss Curve ---
log_history = trainer.state.log_history
train_steps = [h['step'] for h in log_history if 'loss' in h]
train_loss = [h['loss'] for h in log_history if 'loss' in h]
eval_epochs = [h['epoch'] for h in log_history if 'eval_loss' in h]
eval_loss = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(train_steps, train_loss, alpha=0.7, label='Train Loss', color='#2196F3')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')

ax2 = ax1.twiny()
ax2.plot(eval_epochs, eval_loss, 'o-', color='#F44336', label='Val Loss', markersize=8)
ax2.set_xlabel('Epoch')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.set_title('Training & Validation Loss')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_PLOTS}/loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 10e. Summary Metrics Table (copy-paste for slides) ---
print('=' * 50)
print('SCORER MODEL — TEST SET RESULTS')
print('=' * 50)
print(f"{'Metric':<25} {'Value':>10}")
print('-' * 36)
print(f"{'Accuracy':<25} {accuracy_score(labels, preds):>10.3f}")
print(f"{'F1 Macro':<25} {f1_score(labels, preds, average='macro'):>10.3f}")
print(f"{'F1 Weighted':<25} {f1_score(labels, preds, average='weighted'):>10.3f}")
print('-' * 36)
print(f"{'High Recall ★':<25} {per_rec[2]:>10.3f}")
print(f"{'High F1 ★':<25} {per_f1[2]:>10.3f}")
print(f"{'High Precision':<25} {per_prec[2]:>10.3f}")
print('-' * 36)
print(f"{'Medium Recall':<25} {per_rec[1]:>10.3f}")
print(f"{'Medium F1':<25} {per_f1[1]:>10.3f}")
print('-' * 36)
print(f"{'Low Precision':<25} {per_prec[0]:>10.3f}")
print(f"{'Low F1':<25} {per_f1[0]:>10.3f}")
print('=' * 50)
print('\nAll outputs saved to Google Drive:')
print(f'  Plots:   {DRIVE_PLOTS}/')
print(f'  Adapter: {DRIVE_DIR}/scorer_lora_adapter/')

# Save metrics as JSON for programmatic access
metrics_dict = {
    'accuracy': float(accuracy_score(labels, preds)),
    'f1_macro': float(f1_score(labels, preds, average='macro')),
    'f1_weighted': float(f1_score(labels, preds, average='weighted')),
    'recall_high': float(per_rec[2]),
    'f1_high': float(per_f1[2]),
    'precision_high': float(per_prec[2]),
    'recall_medium': float(per_rec[1]),
    'f1_medium': float(per_f1[1]),
    'precision_low': float(per_prec[0]),
    'f1_low': float(per_f1[0]),
}
with open(f'{DRIVE_DIR}/test_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f'  Metrics: {DRIVE_DIR}/test_metrics.json')

## 11. Save LoRA Adapter

In [ ]:
ADAPTER_DIR = '/content/scorer_lora_adapter'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"LoRA adapter saved to {ADAPTER_DIR}")
print(f"Files: {os.listdir(ADAPTER_DIR)}")

# Save to Drive for persistence
DRIVE_ADAPTER = f'{DRIVE_DIR}/scorer_lora_adapter'
os.makedirs(DRIVE_ADAPTER, exist_ok=True)
model.save_pretrained(DRIVE_ADAPTER)
tokenizer.save_pretrained(DRIVE_ADAPTER)
print(f"Also saved to Drive: {DRIVE_ADAPTER}")

## 12. Inference Example

How to load and use the trained model for scoring new issues.

In [ ]:
from peft import PeftModel

# --- Load for inference ---
# base_model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_ID, num_labels=3, torch_dtype=torch.bfloat16, trust_remote_code=True
# )
# model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
# model.eval()

BAND_NAMES = ['low', 'medium', 'high']
BAND_SCORES = [0.175, 0.50, 0.825]  # midpoints of each band


def score_issue(text: str) -> dict:
    """Score a single issue snapshot.
    
    Returns: {band: int, band_name: str, score: float, probabilities: list}
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=MAX_SEQ_LEN, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    pred_band = int(torch.argmax(logits, dim=-1).item())
    return {
        'band': pred_band,
        'band_name': BAND_NAMES[pred_band],
        'score': BAND_SCORES[pred_band],
        'probabilities': {name: round(p, 4) for name, p in zip(BAND_NAMES, probs)},
    }


# Test on a few examples from test set
for i in [0, len(test_data)//2, -1]:
    result = score_issue(test_data[i]['text'])
    actual = test_data[i]['label']
    print(f"Actual: {BAND_NAMES[actual]:8s} | Predicted: {result['band_name']:8s} | "
          f"Probs: {result['probabilities']}")